### 목표 설정
- 음악 데이터로 트렌드를 읽는 똑똑한 음악 소비자가 되기
- **[배경 이야기]**
  음악을 좋아하는 동연씨는 멜론에서 최신 인기곡을 많이 듣습니다.
  최근 동아리에서 사회자 역할을 맡아서 모임 진행중에 동아리원들이 좋아할 음악을 선곡하는 역할을 맡았습니다.
  하지만 매번 멜론 앱에서 일일이 차트를 확인하고, 선곡을 준비하는데 시간이 많이 걸리고 있습니다.

  한편 음악 프로듀서를 꿈꾸는 상원씨는 요즘 대중이 어떤 음악을 좋아하는지 데이터로 분석해서 새로운 곡을 작곡하고 싶어한다. 두 사람은 함께 멜론 차트 데이터를 수집해서 대중음악 트렌드를 분석하고 활용하기로 했습니다.

### BX / CX / DX 분석(DCX)
#### BX(Brand Experience) : 멜론의 브랜드와 음악 트렌드의 신뢰성
- 멜론은 국내 대표 음악 스트리밍 플랫폼 / 실시간 차트를 통해 대중 음악 트렌드를 가장 빠르게 보여줌
#### CX(Customer Experience) : 사용자 맞춤형 음악 데이터를 활용
- 매일 실시간으로 바뀌는 인기 차트를 자동으로 수집해서 사용자 경험 개선
#### DX(Digital Experience) : 자동화된 음악 데이터 수집과 시각화
- Requests 와 BeautifulSoup을 사용해서 멜론 Top100 차트를 자동으로 크롤링하고 분석

In [13]:
# 필요한 라이브러리 가져오기
import requests as req
from bs4 import BeautifulSoup as bs
import pandas as pd

In [14]:
# 1. req를 활용해서 멜론차트 사이트를 요청하자
req.get("https://www.melon.com/chart/index.htm")

<Response [406]>

In [ ]:
# 신분증(header)이 없기 때문에 컴퓨터로 인식하고 접근차단
# 응답코드 400번대 -> 아예 입장 불가 접근차단
# 응답코드 200번대 -> 출력값이 보안이 걸려 이상한 문자로 출력됨

### 응답코드가 400번대가 나왔다?
- 여기서는 요청의 문제(서버가 헤더를 읽고 요청을 거절하는 방식을 선택)
- 브라우저가 아닌 코드를 통해 요청 -> 헤더정보가 누락된 채로 요청 -> 서버가 요청을 거절
- 브라우저인 척 속이는 작업이 필요 -> 브라우저 정보를 담아서 보내자

In [15]:
# 브라우저를 통해 요청하는 것처럼 작업 -> http 요청
headers = {"User-Agent" : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36"}

In [16]:
# 다시 요청을 보내보기(이때 header라는 신분증을 함께 보내주기)
res = req.get("https://www.melon.com/chart/index.htm", headers = headers)
res

<Response [200]>

In [17]:
res.text

'<!DOCTYPE html>\r\n<html lang="ko">\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n<head>\r\n\t\t\r\n\t\r\n\t\r\n\t\r\n\t\r\n\r\n\t\r\n\r\n\t\r\n\t\r\n\t\r\n\t\r\n\r\n\t<meta http-equiv="Content-Type" content="text/html; charset=utf-8"/>\r\n\t<meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1" />\r\n\r\n\t\r\n\r\n\t\r\n\r\n\t<title>멜론차트>TOP100>멜론</title>\r\n\t<meta name="keywords" content="음악서비스, 멜론차트, 멜론TOP100, 최신음악, 인기가요, 뮤직비디오, 앨범, 플레이어, 스트리밍, 다운로드, 아티스트플러스, 아티스트채널" />\r\n\t<meta name="description" content="No.1 뮤직플랫폼 멜론! 최신 트렌드부터 나를 아는 똑똑한 음악추천까지!" />\r\n\t<meta name="naver-site-verification" content="ee85ff6db1fa8f2226bcb671ecb2bcdbcffb6f8b" />\r\n\t<meta name="google-site-verification" content="q4tbTQhmxa4La3OdNLsNOCxrJ_WV6lUlBFrFW4-HqQc" />\r\n\t<meta property="fb:app_id" content="4022717807957185"/>\r\n\t<meta property="og:title" content="Melon"/>\r\n\t<meta property="og:image" content="https://cdnimg.melon.co.kr/resource/image/web/common/logo_melon142x99.png

In [18]:
# 데이터 파싱 str -> html 으로 변환
soup = bs(res.text, 'lxml')
soup

<!DOCTYPE html>
<html lang="ko">
<head>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="IE=edge,chrome=1" http-equiv="X-UA-Compatible"/>
<title>멜론차트&gt;TOP100&gt;멜론</title>
<meta content="음악서비스, 멜론차트, 멜론TOP100, 최신음악, 인기가요, 뮤직비디오, 앨범, 플레이어, 스트리밍, 다운로드, 아티스트플러스, 아티스트채널" name="keywords"/>
<meta content="No.1 뮤직플랫폼 멜론! 최신 트렌드부터 나를 아는 똑똑한 음악추천까지!" name="description"/>
<meta content="ee85ff6db1fa8f2226bcb671ecb2bcdbcffb6f8b" name="naver-site-verification"/>
<meta content="q4tbTQhmxa4La3OdNLsNOCxrJ_WV6lUlBFrFW4-HqQc" name="google-site-verification"/>
<meta content="4022717807957185" property="fb:app_id"/>
<meta content="Melon" property="og:title"/>
<meta content="https://cdnimg.melon.co.kr/resource/image/web/common/logo_melon142x99.png" property="og:image"/>
<meta content="음악이 필요한 순간, 멜론" property="og:description"/>
<meta content="http://www.melon.com/chart/index.htm" property="og:url"/>
<meta content="website" property="og:type"/>
<meta content="멜론" proper

In [ ]:
# #lst50 > td:nth-child(6) > div > div > div.ellipsis.rank01 > span > a
# #lst50 클래스선택자 지움 -> 노래제목 100개 필요
# :nth-child(6) 지움 -> 노래제목 1개만 선택하는 것이 아니기 때문

In [19]:
# 노래제목 수집
# div.ellipsis.rank01 -> 노래제목이 들어있는 클래스 선택자
title = soup.select('td > div > div > div.ellipsis.rank01 > span > a')
len(title)

100

In [20]:
# 실습
# 가수 이름 수집
name = soup.select('td > div > div > div.ellipsis.rank02 > a')
len(name)

122

In [21]:
# #lst50 > td:nth-child(6) > div > div > div.ellipsis.rank02 > a
# 가수가 2명 이상인 경우 가수명을 각각 가져오기 때문에 갯수가 맞지않음
# 해결방법 : a태그를 직접적으로 가져오는 것이 아닌 숨어있던 span태그를 찾아서 
# 묶음으로 데이터를 가져오기
singer = name = soup.select('td > div > div > div.ellipsis.rank02 > span')
len(singer)

100

In [22]:
# 실습
# 순위 수집
rank = soup.select('td > div > span.rank')
len(rank)

100

In [38]:
# 실습
# 가수이름 / 노래제목 / 순위를 추출해서 각각 컨텐츠만 담아주는 작업
# 1. 각 컨텐츠가 담기는 빈 리스트 만들어주기
# 2. 반복문을 통해서 각 리스트에 컨텐츠만 추출하기
# 힌트 : text함수를 사용해서 컨텐츠만 추출.

singer_list = []
title_list = []
rank_list = []

# 위 3개의 변수 인덱스 값을 체크해보자 : 0 ~ 99 까지 인덱스번호를 공통으로 사용
# singer_list, title_list, rank_list 개수가 모두 100개로 동일
for i in range(len(rank)): # 0 ~ 99 인덱스 만큼 반복문을 돌리겠다.
    singer_list.append(singer[i].text)
    title_list.append(title[i].text)
    rank_list.append(rank[i].text)


In [46]:
# 실습
# 데이터 프레임 제작
# 딕셔너리 형태로 만들어서 데이터 프레임 제작
# 컬럼명은 순위 / 노래제목 / 가수이름으로 지정 -> 딕셔너리의 key 잘 잡아주기
# 인덱스 자리는 순위로 대체 / 힌트 : set_index() 함수 활용
# set_index() 함수 -> 원본에 반영 되지않음
# 변수에 다시 저장하거나, inplace옵션을 True로 적용해줘야한다.
data = {'순위' : rank_list,
        '노래제목' : title_list,
        '가수이름' : singer_list
       }
df = pd.DataFrame(data)
df.set_index('순위', inplace = True)
df

,노래제목,가수이름
순위,,
1,404 (New Era),KiiiKiii (키키)
2,BANG BANG,IVE (아이브)
3,Good Goodbye,화사 (HWASA)
4,그대 작은 나의 세상이 되어,카더가든
5,사랑하게 될 거야,한로로
...,...,...
96,나였으면,"나윤권, 도경수(D.O.)"
97,Dirty Work,aespa
98,슬픈 초대장,순순희(지환)


### 이번 파일 정리
1. req로 요청을 보냈을때 400번대 응답이 나오면 브라우저 정보 담아보내기
2. 데이터 수집 할 때 수집된 데이터 갯수 항상 검증하기
3. 만약 갯수가 다르다면, 데이터프레임 / 파일을 만들 때 오류가 발생 -> 주변 태그 분석해 수집
4. 데이터를 수집하고 활용 할때 사용자에게 적합한 형태로 만들기

### 추가 아이디어
1. 멜론차트 뿐만아니라 스포티파이, 유투브뮤직 차트를 비교분석해 글로벌 음악 트렌드 파악
2. 멜론 상위 100곡의 가사 데이터를 기반으로 감성분석(긍정/부정)
3. 인기 아티스트에 순위 변동 추이를 분석해서 음악 시장의 변화 예측